In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/wine-and-food-pairing-dataset/wine_food_pairings.csv


In [3]:
# !pip install -q \
#     transformers==4.44.2 \
#     peft==0.13.2 \
#     bitsandbytes==0.43.1 \
#     accelerate==0.33.0 \
#     trl==0.9.6 \
#     datasets==2.20.0 \
#     pyarrow==19.0.0


!pip install -q \
    bitsandbytes==0.43.1 \
    triton==2.1.0 \
    transformers==4.44.2 \
    peft==0.13.2 \
    accelerate==0.33.0 \
    trl==0.9.6 \
    datasets==2.20.0 \
    pyarrow==19.0.0



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 70.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 14.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 40.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.7 MB/s eta 0:00:00
ERRO

In [ ]:
# import os
# os._exit(00)

In [1]:
# =============================
# Kaggle QLoRA Fine-Tune (Wine–Food Pairing)
# =============================

# !pip install -q bitsandbytes transformers peft accelerate datasets trl


import torch
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig

print("✅ torch:", torch.__version__)
print("✅ bitsandbytes:", bnb.__version__)
print("✅ CUDA available:", torch.cuda.is_available())

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
import pandas as pd

# 1️⃣ Load your dataset (Kaggle input dataset path)
df = pd.read_csv("/kaggle/input/wine-and-food-pairing-dataset/wine_food_pairings.csv")

# 2️⃣ Prepare training text format (Option A: recommendation mode)
def build_prompt(row):
    return (
        f"Food: {row['food_item']}\n"
        f"Cuisine: {row['cuisine']}\n"
        f"Task: Recommend a suitable wine and explain.\n"
        f"Answer: {row['description']}"
    )

df["text"] = df.apply(build_prompt, axis=1)
df.head()

2025-10-22 12:40:58.043521: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761136858.066913     131 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761136858.073898     131 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


,wine_type,wine_category,food_item,food_category,cuisine,pairing_quality,quality_label,description,text
0,Syrah/Shiraz,Red,smoked sausage,Smoky BBQ,Spanish,2,Poor,Heuristic pairing assessment.,Food: smoked sausage\nCuisine: Spanish\nTask: ...
1,Grenache,Red,charcuterie board,Salty Snack,French,3,Neutral,Heuristic pairing assessment.,Food: charcuterie board\nCuisine: French\nTask...
2,Madeira,Fortified,lemon tart,Dessert,French,4,Good,Acidic wine balances acidic food.,Food: lemon tart\nCuisine: French\nTask: Recom...
3,Cabernet Sauvignon,Red,roast lamb,Red Meat,Mexican,5,Excellent,Tannic red complements red meat fat.,Food: roast lamb\nCuisine: Mexican\nTask: Reco...
4,Viognier,White,duck à l’orange,Poultry,Vietnamese,2,Poor,Heuristic pairing assessment.,Food: duck à l’orange\nCuisine: Vietnamese\nTa...


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34933 entries, 0 to 34932
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   wine_type        34933 non-null  object
 1   wine_category    34933 non-null  object
 2   food_item        34933 non-null  object
 3   food_category    34933 non-null  object
 4   cuisine          34933 non-null  object
 5   pairing_quality  34933 non-null  int64 
 6   quality_label    34933 non-null  object
 7   description      34933 non-null  object
 8   text             34933 non-null  object
dtypes: int64(1), object(8)
memory usage: 2.4+ MB


In [ ]:
# 3️⃣ Wrap into Hugging Face dataset
from datasets import Dataset
dataset = Dataset.from_pandas(df[["text"]])
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# 4️⃣ Tokenize
model_name = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# def preprocess(examples):
#     return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

# tokenized = dataset.map(preprocess, batched=True, remove_columns=["text"])


def preprocess(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized = dataset.map(preprocess, batched=True, remove_columns=["text"])


# 5️⃣ Model + QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✅ Model config done!")

In [ ]:
print("✅ Starting Training...")
# 6️⃣ Training
training_args = TrainingArguments(
    output_dir="./wine_qlora",
    report_to="none",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"]
)

trainer.train()
print("✅ Training done!")

In [ ]:
# 7️⃣ Save adapter
model.save_pretrained("./wine_qlora_adapter")
tokenizer.save_pretrained("./wine_qlora_adapter")